# TRACE-RAG: end-to-end Colab TPU run

This notebook checks the project, exercises the **PyTorch/XLA TPU paths** for Hugging Face embeddings and NLI verification, runs the full mini-corpus trust stream, and saves presentation-friendly artifacts. An optional cell downloads a gold-preserving Natural Questions (NQ) subset.

**Read the scope before presenting results:** the default answer backend is the deterministic `StubLLM`; its answers are not a quality benchmark. The mini corpus is only an integration smoke test. To make answer-quality claims, use a real, documented answer model and a real benchmark run. TPU execution is experimental until this notebook completes successfully in your own Colab TPU runtime.

TPU acceleration applies to the configured transformer inference only. Parsing, SQLite/provenance, NumPy retrieval/indexing, signal computation and most trust bookkeeping stay on CPU. The first XLA compilation can be slow; TPU does not guarantee lower end-to-end wall time for small runs. Use the recorded timings rather than assuming a speedup.

## Before running

1. In Colab choose **Runtime → Change runtime type → TPU**.
2. Run every cell from top to bottom in a fresh runtime.
3. Colab supplies a compatible `torch`/`torch_xla` pair. This notebook deliberately does **not** install or upgrade either package; arbitrary XLA/PyTorch versions are a common cause of TPU startup failures.
4. A working network connection is needed for the public Hugging Face model weights. No access token or API key is required for the default stub-backed mini run.

The notebook clones the fixed Arena branch below. It never embeds GitHub credentials.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import time
import datetime
import csv
import zipfile
import platform
import importlib.metadata

REPO_URL = "https://github.com/athishss/-Adaptive-Trust-Based-Framework-for-Detecting-and-Preventing-Knowledge-Base-Poisoning-in-RAG-Systems.git"
BRANCH = "arena/aac1ce08-adaptive-trust-based-framework"
REPO = Path("/content/trace-rag")

if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f"{REPO} exists but is not a Git checkout; choose an empty path.")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
         REPO_URL, str(REPO)], check=True,
    )
else:
    current_branch = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True,
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f"Expected branch {BRANCH!r}, found {current_branch!r}.")

os.chdir(REPO)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repo:", REPO)
print("branch:", BRANCH)
print("commit:", GIT_COMMIT)

In [ ]:
# Probe XLA in a short-lived process so the notebook kernel does not retain
# a TPU client while the project CLI launches its own process later.
probe_code = (
    "import torch; import torch_xla; import torch_xla.core.xla_model as xm; "
    "print('torch:', torch.__version__); "
    "print('torch_xla:', getattr(torch_xla, '__version__', 'installed')); "
    "print('xla_device:', xm.xla_device())"
)
probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True)
print(probe.stdout)
if probe.returncode:
    print(probe.stderr)
    raise RuntimeError(
        "No working PyTorch/XLA device was detected. Switch Colab to a TPU runtime "
        "and restart the notebook; do not pip-install a random torch_xla wheel."
    )
if "xla:" not in probe.stdout.lower():
    raise RuntimeError(f"The detected device is not XLA/TPU: {probe.stdout}")
TPU_PROBE = probe.stdout.strip()

In [ ]:
%pip -q install -e ".[data,plots,dev]" "transformers>=4.40" sentencepiece
print("Installed project/data/test dependencies; the Colab torch/torch_xla pair was left untouched.")

## 1. Run the repository tests

This is a software regression check, not a benchmark. Optional FAISS/model tests may skip when their extra dependencies are not installed.

In [ ]:
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path("runs/colab_tpu") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

suite = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-ra"],
    cwd=REPO, capture_output=True, text=True,
)
TEST_LOG = RUN_DIR / "pytest_output.txt"
TEST_LOG.write_text(suite.stdout + "\n" + suite.stderr, encoding="utf-8")
print(suite.stdout)
if suite.stderr:
    print(suite.stderr)
if suite.returncode:
    raise RuntimeError(f"pytest failed; full log saved to {TEST_LOG}")
print("All non-skipped tests passed. Log:", TEST_LOG)

## 2. End-to-end TPU mini smoke

The default run uses a small public MiniLM encoder on XLA, DeBERTa NLI on XLA, the repository's chronological trust stream, and `StubLLM` for deterministic/offline answer generation. `ANSWER_BACKEND = "huggingface"` opts into a small local answer model on TPU; generation can be slow and is not validated by the project test suite. Do not mix results from different answer backends.

In [ ]:
SEED = 20260921
NLI_MAX_LENGTH = 256
ANSWER_BACKEND = "stub"  # change to "huggingface" only if you want a local TPU answer model
ANSWER_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

os.environ["TRACE_RAG_NLI_DEVICE"] = "tpu"
os.environ["TRACE_RAG_NLI_MAX_LENGTH"] = str(NLI_MAX_LENGTH)

MINI_OUT = RUN_DIR / "mini"
command = [
    sys.executable, "scripts/run_trust_stream.py",
    "--config", "config/default.yaml",
    "--corpus", "examples/mini_corpus.jsonl",
    "--queries", "examples/mini_questions.json",
    "--steps", "12", "--targets", "2", "--seed", str(SEED),
    "--out", str(MINI_OUT),
    "--set", "embedding.backend=huggingface",
    "--set", "embedding.model_name=sentence-transformers/all-MiniLM-L6-v2",
    "--set", "embedding.device=tpu",
    "--set", "embedding.batch_size=4",
    "--set", "embedding.max_length=128",
    "--set", "trust.nli_mode=nli",
    "--set", f"generation.backend={ANSWER_BACKEND}",
]
if ANSWER_BACKEND == "huggingface":
    command.extend([
        "--set", f"generation.model_name={ANSWER_MODEL}",
        "--set", "generation.device=tpu",
        "--set", "generation.max_tokens=96",
    ])

run_env = os.environ.copy()
started = time.perf_counter()
mini_run = subprocess.run(command, cwd=REPO, env=run_env, capture_output=True, text=True)
mini_seconds = time.perf_counter() - started
(MINI_OUT / "runner.log").write_text(mini_run.stdout + "\n" + mini_run.stderr,
                                      encoding="utf-8")
print(mini_run.stdout[-12000:])
if mini_run.returncode:
    print(mini_run.stderr[-6000:])
    raise RuntimeError(f"mini stream failed; full output saved to {MINI_OUT / 'runner.log'}")

MINI_METRICS = json.loads((MINI_OUT / "metrics.json").read_text(encoding="utf-8"))
MINI_METRICS["notebook_wall_seconds"] = round(mini_seconds, 3)
assert MINI_METRICS["verifier_mode"] == "nli", MINI_METRICS
assert MINI_METRICS["embedding_device"].startswith("xla"), MINI_METRICS
assert MINI_METRICS["verifier_device"].startswith("xla"), MINI_METRICS
print("Confirmed XLA embedding and NLI devices:",
      MINI_METRICS["embedding_device"], MINI_METRICS["verifier_device"])
print("Mini metrics:", json.dumps({k: MINI_METRICS[k] for k in (
    "poison_retrieved_rate", "poison_citation_rate", "poison_quarantined",
    "clean_false_quarantine", "mean_latency_ms", "mean_llm_calls_per_query",
)}, indent=2))
RUN_METRICS = [{"dataset": "mini-corpus smoke", "metrics": MINI_METRICS}]

## 3. Optional Natural Questions subset

Set `RUN_NQ_SUBSET = True` to download the BEIR NQ files and build a deterministic 20,000-passage, gold-preserving subset for 100 queries. The downloader fetches the source corpus before subset creation, so budget for the full source download, disk, network time, model-cache space, and TPU compilation. This stream runner is still a smoke test—not the complete attack × baseline × seed evaluation or a paper headline result.

The provided `config/nq_tpu.yaml` keeps Contriever and the NLI verifier on TPU/XLA; NumPy retrieval/indexing and SQLite remain on CPU. For comparable research runs, use the same model, max length, subset, config, seed protocol and answer backend across all methods.

In [ ]:
RUN_NQ_SUBSET = False
NQ_DIR = Path("data/nq")
NQ_OUT = RUN_DIR / "nq_subset_20k"
NQ_METRICS = None

if RUN_NQ_SUBSET:
    download = subprocess.run(
        [sys.executable, "scripts/download_data.py", "--dataset", "nq",
         "--out", str(NQ_DIR), "--subset", "20000", "--queries", "100",
         "--seed", str(SEED)],
        cwd=REPO, capture_output=True, text=True,
    )
    print(download.stdout[-10000:])
    if download.returncode:
        print(download.stderr[-5000:])
        raise RuntimeError("NQ preparation failed; see output above.")

    nq_command = [
        sys.executable, "scripts/run_trust_stream.py",
        "--config", "config/nq_tpu.yaml",
        "--corpus", str(NQ_DIR / "corpus_subset.parquet"),
        "--queries", str(NQ_DIR / "queries_subset.jsonl"),
        "--qrels", str(NQ_DIR / "qrels.tsv"),
        "--steps", "40", "--targets", "5", "--seed", str(SEED),
        "--out", str(NQ_OUT),
        "--set", "embedding.batch_size=4",
        "--set", f"generation.backend={ANSWER_BACKEND}",
    ]
    if ANSWER_BACKEND == "huggingface":
        nq_command.extend([
            "--set", f"generation.model_name={ANSWER_MODEL}",
            "--set", "generation.device=tpu",
            "--set", "generation.max_tokens=96",
        ])
    nq_run = subprocess.run(nq_command, cwd=REPO, env=os.environ.copy(),
                            capture_output=True, text=True)
    (NQ_OUT / "runner.log").write_text(nq_run.stdout + "\n" + nq_run.stderr,
                                        encoding="utf-8")
    print(nq_run.stdout[-12000:])
    if nq_run.returncode:
        print(nq_run.stderr[-6000:])
        raise RuntimeError(f"NQ stream failed; full output saved to {NQ_OUT / 'runner.log'}")
    NQ_METRICS = json.loads((NQ_OUT / "metrics.json").read_text(encoding="utf-8"))
    RUN_METRICS.append({"dataset": "NQ 20k subset smoke", "metrics": NQ_METRICS})
else:
    print("Skipped NQ download/run. Set RUN_NQ_SUBSET=True and rerun this cell to opt in.")

## 4. Optional embedding timing probe

This runs one CPU and one TPU MiniLM batch in separate subprocesses, records model-load, first-batch warm-up, steady-state time, and throughput, then releases each process before continuing. These timings are indicative only: TPU uses static padding and incurs XLA synchronization/compilation. Repeat the run with matched shapes, batch sizes, multiple seeds/runs, and report warm-up separately before claiming a speedup.

In [ ]:
benchmark_code = "\n".join([
    "import gc, json, time",
    "from trace_rag.embeddings.hf_embedder import HFEmbedder",
    "model_name = 'sentence-transformers/all-MiniLM-L6-v2'",
    "texts = [('The Eiffel Tower is a landmark in Paris, designed and built in the late nineteenth century. ') * 5 for _ in range(64)]",
    "def measure(device):",
    "    t0 = time.perf_counter()",
    "    embedder = HFEmbedder(model_name=model_name, device=device, max_length=128)",
    "    load_s = time.perf_counter() - t0",
    "    t0 = time.perf_counter()",
    "    warm = embedder.encode_documents(texts[:4], batch_size=4)",
    "    warmup_s = time.perf_counter() - t0",
    "    t0 = time.perf_counter()",
    "    vectors = embedder.encode_documents(texts, batch_size=4)",
    "    steady_s = time.perf_counter() - t0",
    "    result = {'requested_device': device, 'resolved_device': str(embedder.device), 'load_seconds': round(load_s, 3), 'warmup_seconds': round(warmup_s, 3), 'steady_seconds': round(steady_s, 3), 'documents': len(texts), 'documents_per_second': round(len(texts) / max(steady_s, 1e-9), 3), 'output_shape': list(vectors.shape), 'output_finite': bool(__import__('numpy').isfinite(vectors).all())}",
    "    del embedder, vectors, warm",
    "    gc.collect()",
    "    return result",
    "results = {'cpu': measure('cpu'), 'tpu': measure('tpu')}",
    "results['comparison_note'] = 'Indicative only: XLA uses static 128-token padding and includes accelerator synchronization; CPU padding is dynamic. Repeat with matched shapes/batches and multiple runs before making a speedup claim.'",
    "print('BENCHMARK_JSON=' + json.dumps(results))",
])
bench = subprocess.run([sys.executable, "-c", benchmark_code], cwd=REPO,
                        capture_output=True, text=True)
print(bench.stdout)
if bench.returncode:
    print(bench.stderr[-6000:])
    raise RuntimeError("CPU/TPU embedding throughput probe failed.")
benchmark_line = next((line for line in bench.stdout.splitlines() if line.startswith("BENCHMARK_JSON=")), None)
if benchmark_line is None:
    raise RuntimeError("Benchmark process did not return its JSON marker.")
EMBEDDING_BENCHMARK = json.loads(benchmark_line.split("=", 1)[1])
(RUN_DIR / "embedding_benchmark.json").write_text(
    json.dumps(EMBEDDING_BENCHMARK, indent=2), encoding="utf-8",
)

## 4. Save reproducibility metadata and presentation artifacts

The report includes the resolved config, raw smoke metrics, stream JSONL, trust history, run logs, and test log. It intentionally labels the mini/NQ stream outputs as smoke tests; do not turn these into a claim about production or general benchmark performance.

In [ ]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"

summary_fields = [
    "dataset", "seed", "embedding_model", "embedding_device", "verifier_mode",
    "verifier_device", "generation_backend", "generation_model", "steps",
    "poison_retrieved_rate", "poison_citation_rate", "poison_quarantined",
    "clean_false_quarantine", "mean_latency_ms", "mean_llm_calls_per_query",
]
summary_rows = []
for run in RUN_METRICS:
    row = {"dataset": run["dataset"]}
    row.update({key: run["metrics"].get(key) for key in summary_fields if key != "dataset"})
    summary_rows.append(row)

summary_path = RUN_DIR / "presentation_summary.csv"
with summary_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=summary_fields)
    writer.writeheader()
    writer.writerows(summary_rows)

metadata = {
    "title": "TRACE-RAG Colab TPU smoke run",
    "run_id_utc": RUN_ID,
    "git_branch": BRANCH,
    "git_commit": GIT_COMMIT,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": package_version("torch"),
    "torch_xla": package_version("torch-xla"),
    "transformers": package_version("transformers"),
    "pyarrow": package_version("pyarrow"),
    "tpu_probe": TPU_PROBE,
    "seed": SEED,
    "nli_device_request": os.environ.get("TRACE_RAG_NLI_DEVICE"),
    "embedding_benchmark": EMBEDDING_BENCHMARK,
    "nli_max_length": NLI_MAX_LENGTH,
    "answer_backend": ANSWER_BACKEND,
    "answer_model": ANSWER_MODEL if ANSWER_BACKEND == "huggingface" else "stub",
    "scope_warning": (
        "Mini/NQ stream runs are component smoke tests, not production validation or "
        "headline benchmark results. StubLLM output is not answer-quality evidence."
    ),
    "runs": [{"dataset": item["dataset"], "metrics": item["metrics"]}
             for item in RUN_METRICS],
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(RUN_DIR / "RESULTS_README.md").write_text(
    "# TRACE-RAG Colab TPU artifacts\n\n"
    "These are reproducibility and integration-smoke artifacts, not a production-readiness "
    "certification or a complete benchmark. See `run_metadata.json` for the code revision, "
    "models, devices, seeds and scope limitations. The mini-corpus is not representative.\n",
    encoding="utf-8",
)

archive = RUN_DIR.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            zipped.write(path, path.relative_to(RUN_DIR.parent))

print("Summary CSV:", summary_path)
print("Metadata:", RUN_DIR / "run_metadata.json")
print("Results archive:", archive)
print(summary_path.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(archive)))
print("Optional: download this ZIP or copy it into Google Drive using the next cell.")

In [ ]:
# Optional persistence to Google Drive. Set SAVE_TO_DRIVE=True before running.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/TRACE-RAG") / archive.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, destination)
    print("Copied results to:", destination)

## Reading and presenting the output

- Report the exact `git_commit`, config, data subset, random seed, model IDs, resolved devices, and answer backend from `run_metadata.json`.
- Compare timing only under the same hardware, model, batch size, max length, corpus, and warm-up protocol. Separate first-compile time from steady-state TPU time if you add a dedicated benchmark.
- `StubLLM` makes the full flow reproducible, but its answer text is lexical placeholder behavior. It cannot support answer-accuracy claims.
- `poison_quarantined`, `clean_false_quarantine`, and citation rates from a mini/NQ stream are smoke metrics, not attack-success-rate estimates from a controlled benchmark matrix.
- The project is a research prototype, **not production-ready for company deployment**. See `docs/PRODUCTION_READINESS.md` for required security, privacy, reliability, and validation work.